# OmniVoice Quick Start

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/k2-fsa/OmniVoice/blob/master/docs/OmniVoice.ipynb)

This notebook demonstrates the basic usage of [OmniVoice](https://github.com/k2-fsa/OmniVoice), a massively multilingual zero-shot TTS model supporting 600+ languages.

**Contents:**
1. Installation
2. Option A — Gradio Demo (interactive web UI, no code needed)
3. Option B — Python API
   - 3.1 Load Model
   - 3.2 Voice Cloning
   - 3.3 Voice Design
   - 3.4 Auto Voice

## 1. Installation

Colab already provides a compatible PyTorch + CUDA environment, so we only need to install OmniVoice.

In [ ]:
!pip install omnivoice

## 2. Option A — Gradio Demo

Launch an interactive web UI with a public Gradio link. The `--share` flag creates a temporary public URL so you can access the demo from any browser.

> **If you prefer to use the Python API directly, skip to Option B below.**

In [ ]:
!omnivoice-demo --share

## 3. Option B — Python API

### 3.1 Load Model

In [ ]:
from omnivoice import OmniVoice
import soundfile as sf
import torch
from IPython.display import Audio, display

model = OmniVoice.from_pretrained(
    "k2-fsa/OmniVoice",
    device_map="cuda:0",
    dtype=torch.float16,
    load_asr=True,
)

### 3.2 Voice Cloning

Clone a voice from a short (3-10s) reference audio clip. Upload your own `ref.wav` or use any audio file.

`ref_text` is optional — if omitted, the model uses Whisper ASR to auto-transcribe it.

In [ ]:
from google.colab import files

print("Upload a reference audio file (wav/mp3/flac):")
uploaded = files.upload()
ref_audio_path = list(uploaded.keys())[0]
print(f"Uploaded: {ref_audio_path}")

In [ ]:
audio = model.generate(
    text="Hello, this is a test of zero-shot voice cloning.",
    ref_audio=ref_audio_path,
    # ref_text="Transcription of the reference audio.",  # optional
)

sf.write("clone_out.wav", audio[0], 24000)
display(Audio(audio[0], rate=24000))

### 3.3 Voice Design

Describe the desired voice with speaker attributes — no reference audio needed.

Supported attributes: gender, age, pitch, style (whisper), English accent, Chinese dialect. See [docs/voice-design.md](https://github.com/k2-fsa/OmniVoice/blob/master/docs/voice-design.md) for the full list.

In [ ]:
audio = model.generate(
    text="Hello, this is a test of zero-shot voice design.",
    instruct="female, low pitch, british accent",
)

sf.write("design_out.wav", audio[0], 24000)
display(Audio(audio[0], rate=24000))

### 3.4 Auto Voice

Let the model choose a voice automatically — no reference audio or instruct needed.

In [ ]:
audio = model.generate(
    text="This is a sentence generated with automatic voice selection.",
)

sf.write("auto_out.wav", audio[0], 24000)
display(Audio(audio[0], rate=24000))

In [4]:
from google.colab import drive
drive.mount('/content/drive')

# Extraction ultra-rapide depuis Drive vers le disque local de Colab (~30 secondes)
!mkdir -p /content/kabyle_data
!tar -xf "/content/drive/MyDrive/Kabyle_Dataset/kabyle_cv27.tar" -C /content/kabyle_data

path = "/content/kabyle_data"
print("Dataset prêt dans :", path)

Mounted at /content/drive
Dataset prêt dans : /content/kabyle_data


In [5]:
import os
import pandas as pd
from datasets import Dataset, Audio

path = "/content/kabyle_data"

# 1. Charger validated.tsv
tsv_path = None
for root, dirs, files in os.walk(path):
    if "validated.tsv" in files:
        tsv_path = os.path.join(root, "validated.tsv")
        break

df = pd.read_csv(tsv_path, sep="\t", low_memory=False)
sentences_map = dict(zip(df["path"], df["sentence"]))

# 2. Indexer 5 000 audios validés et s'arrêter dès que le quota est atteint
rows = []
for root, dirs, files in os.walk(path):
    for f in files:
        if f.endswith(".mp3") and f in sentences_map:
            rows.append({"audio": os.path.join(root, f), "sentence": str(sentences_map[f])})
            if len(rows) >= 5000:
                break
    if len(rows) >= 5000:
        break

# 3. Créer le dataset Hugging Face (4 500 train / 500 validation) en 16 kHz
raw_ds = Dataset.from_pandas(pd.DataFrame(rows))
raw_ds = raw_ds.cast_column("audio", Audio(sampling_rate=16000))
dataset = raw_ds.train_test_split(test_size=0.1, seed=42)

print(dataset)

DatasetDict({
    train: Dataset({
        features: ['audio', 'sentence'],
        num_rows: 4500
    })
    test: Dataset({
        features: ['audio', 'sentence'],
        num_rows: 500
    })
})


In [10]:
import os
import torch
import pandas as pd
from dataclasses import dataclass
from typing import Any, Dict, List, Union
from datasets import IterableDataset, Features, Audio, Value
from transformers import (
    WhisperProcessor,
    WhisperForConditionalGeneration,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
)
from peft import LoraConfig, get_peft_model

output_dir = "/content/drive/MyDrive/whisper-kabyle-full"
path = "/content/kabyle_data"

# 1. Charger tout validated.tsv (~602 657 phrases) et mélanger l'ordre
tsv_path = None
for root, dirs, files in os.walk(path):
    if "validated.tsv" in files:
        tsv_path = os.path.join(root, "validated.tsv")
        break

df = pd.read_csv(tsv_path, sep="\t", low_memory=False)
df = df.dropna(subset=["path", "sentence"]).sample(frac=1, random_state=42).reset_index(drop=True)
sentences_map = dict(zip(df["path"], df["sentence"]))
print(f"Total de phrases validées disponibles : {len(sentences_map)}")

# 2. Générateur en streaming avec Features déclarées (corrige le TypeError)
def audio_generator():
    for root, dirs, files in os.walk(path):
        for f in files:
            if f.endswith(".mp3") and f in sentences_map:
                yield {
                    "audio": os.path.join(root, f),
                    "sentence": str(sentences_map[f]),
                }

features = Features({
    "audio": Audio(sampling_rate=16000),
    "sentence": Value("string"),
})

stream_ds = IterableDataset.from_generator(audio_generator, features=features)

# Garder les 1 000 premiers pour l'évaluation, tout le reste pour l'entraînement
eval_ds = stream_ds.take(1000)
train_ds = stream_ds.skip(1000)

# 3. Charger Whisper-small avec LoRA (r=32)
model_name = "openai/whisper-small"
processor = WhisperProcessor.from_pretrained(model_name, language="French", task="transcribe")
model = WhisperForConditionalGeneration.from_pretrained(model_name).to("cuda")

model.config.use_cache = False
model.generation_config.language = "french"
model.generation_config.task = "transcribe"
model.generation_config.forced_decoder_ids = None

lora_config = LoraConfig(
    r=32,
    lora_alpha=64,
    target_modules=["q_proj", "v_proj", "k_proj", "out_proj"],
    lora_dropout=0.05,
    bias="none",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# 4. Prétraitement à la volée
def prepare_batch(batch):
    audio = batch["audio"]
    batch["input_features"] = processor.feature_extractor(
        audio["array"], sampling_rate=audio["sampling_rate"]
    ).input_features[0]
    batch["labels"] = processor.tokenizer(batch["sentence"]).input_ids
    return batch

train_ds = train_ds.map(prepare_batch, remove_columns=["audio", "sentence"])
eval_ds = eval_ds.map(prepare_batch, remove_columns=["audio", "sentence"])

# 5. Data Collator
@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any
    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        input_features = [{"input_features": f["input_features"]} for f in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        label_features = [{"input_ids": f["labels"]} for f in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)
        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all().cpu().item():
            labels = labels[:, 1:]
        batch["labels"] = labels
        return batch

data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)

# 6. Entraînement par paliers sauvegardés sur Google Drive
training_args = Seq2SeqTrainingArguments(
    output_dir=output_dir,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,
    learning_rate=5e-4,
    warmup_steps=200,
    max_steps=3000,
    fp16=True,
    eval_strategy="steps",
    eval_steps=500,
    save_steps=500,
    logging_steps=100,
    save_total_limit=2,
    remove_unused_columns=False,
    label_names=["labels"],
    report_to="none",
)

trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    data_collator=data_collator,
    processing_class=processor.feature_extractor,
)

# 7. Reprendre automatiquement depuis le dernier checkpoint s'il existe sur Drive
has_checkpoint = os.path.isdir(output_dir) and any(d.startswith("checkpoint-") for d in os.listdir(output_dir))
trainer.train(resume_from_checkpoint=has_checkpoint)

model.save_pretrained(output_dir)
processor.save_pretrained(output_dir)
print("Modèle sauvegardé dans :", output_dir)

Total de phrases validées disponibles : 602657


Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

trainable params: 7,077,888 || all params: 248,812,800 || trainable%: 2.8447


Step,Training Loss,Validation Loss
3001,0.896497,0.445399


Modèle sauvegardé dans : /content/drive/MyDrive/whisper-kabyle-full


In [9]:
!pip uninstall -y torchao

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0


**Test with: whisper-kabyle-lora**

In [13]:
import torch
import librosa
from google.colab import drive, files
from transformers import WhisperProcessor, WhisperForConditionalGeneration
from peft import PeftModel

# 1. Connecter Google Drive
drive.mount("/content/drive")
lora_dir = "/content/drive/MyDrive/whisper-kabyle-lora"

# 2. Charger Whisper-small + votre adaptateur entraîné
base_model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-small")
model = PeftModel.from_pretrained(base_model, lora_dir).to("cuda")
model.eval()

processor = WhisperProcessor.from_pretrained(lora_dir)

# 3. Importer un de vos fichiers vocaux (.mp3, .wav, .ogg, .m4a)
uploaded = files.upload()
audio_path = list(uploaded.keys())[0]

# 4. Transcrire en kabyle
audio, _ = librosa.load(audio_path, sr=16000)
inputs = processor.feature_extractor(audio, sampling_rate=16000, return_tensors="pt").input_features.to("cuda")

with torch.no_grad():
    predicted_ids = model.generate(inputs, language="french", task="transcribe")

transcription = processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]
print("\n--- Transcription en kabyle ---")
print(transcription)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

Saving Enregistrement.m4a to Enregistrement (1).m4a


/tmp/ipykernel_514/651660601.py:23: UserWarning: PySoundFile failed. Trying audioread instead.
  audio, _ = librosa.load(audio_path, sr=16000)
/usr/local/lib/python3.13/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)



--- Transcription en kabyle ---
Bɣiɣ ad ččeɣ aksem.


**Tst with: whisper-kabyle-full**

In [16]:
import os
import re
import torch
import librosa
import difflib
import pandas as pd
from collections import Counter
from google.colab import drive, files
from transformers import WhisperProcessor, WhisperForConditionalGeneration
from peft import PeftModel

# 1. Charger votre NOUVEAU modèle (whisper-kabyle-full)
drive.mount("/content/drive")
lora_dir = "/content/drive/MyDrive/whisper-kabyle-full"

base_model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-small")
model = PeftModel.from_pretrained(base_model, lora_dir).to("cuda")
model.eval()
processor = WhisperProcessor.from_pretrained(lora_dir)

# 2. Construire le dictionnaire officiel depuis validated.tsv (pour corriger aksem -> aksum)
tsv_path = None
for root, dirs, files_list in os.walk("/content/kabyle_data"):
    if "validated.tsv" in files_list:
        tsv_path = os.path.join(root, "validated.tsv")
        break

df = pd.read_csv(tsv_path, sep="\t", usecols=["sentence"], low_memory=False).dropna()
vocab_counts = Counter()
for s in df["sentence"]:
    words = re.findall(r"[a-zA-ZɛɣḥṛṭẓǧčƐƔḤṚṬẒǦČ'-]+", str(s).lower())
    vocab_counts.update(words)

# Garder les mots validés apparaissant au moins 3 fois dans le dataset
valid_words = {w for w, c in vocab_counts.items() if c >= 3}

def corriger_phrase_kabyle(texte):
    mots = texte.split()
    mots_corriges = []
    for mot in mots:
        # Séparer la ponctuation finale (ex: "aksem." -> "aksem" + ".")
        match = re.match(r"^([^\wɛɣḥṛṭẓǧčƐƔḤṚṬẒǦČ]*)([\wɛɣḥṛṭẓǧčƐƔḤṚṬẒǦČ'-]+)([^\wɛɣḥṛṭẓǧčƐƔḤṚṬẒǦČ]*)$", mot)
        if not match:
            mots_corriges.append(mot)
            continue

        prefix, coeur, suffix = match.groups()
        coeur_lower = coeur.lower()

        # Si le mot existe déjà tel quel dans le dictionnaire kabyle, on le garde
        if coeur_lower in valid_words or len(coeur_lower) <= 2:
            mots_corriges.append(mot)
        else:
            # Chercher les mots les plus proches dans le dataset (ex: aksem -> aksum)
            candidats = difflib.get_close_matches(coeur_lower, valid_words, n=5, cutoff=0.75)
            if candidats:
                # Choisir le candidat le plus fréquent dans la langue kabyle
                meilleur = max(candidats, key=lambda w: vocab_counts[w])
                if coeur[0].isupper():
                    meilleur = meilleur.capitalize()
                mots_corriges.append(f"{prefix}{meilleur}{suffix}")
            else:
                mots_corriges.append(mot)

    return " ".join(mots_corriges)

# 3. Importer votre fichier vocal et transcrire avec Beam Search (num_beams=5)
uploaded = files.upload()
audio_path = list(uploaded.keys())[0]

audio, _ = librosa.load(audio_path, sr=16000)
inputs = processor.feature_extractor(audio, sampling_rate=16000, return_tensors="pt").input_features.to("cuda")

with torch.no_grad():
    predicted_ids = model.generate(
        inputs,
        language="french",
        task="transcribe",
        num_beams=5,           # Explore 5 hypothèses pour plus de précision
        repetition_penalty=1.1
    )

transcription_brute = processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]
phrase_corrigee = corriger_phrase_kabyle(transcription_brute)

print("\n--- Transcription brute Whisper ---")
print(transcription_brute)
print("\n--- Phrase complète corrigée ---")
print(phrase_corrigee)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

Saving nn.m4a to nn.m4a


/tmp/ipykernel_514/3973676284.py:71: UserWarning: PySoundFile failed. Trying audioread instead.
  audio, _ = librosa.load(audio_path, sr=16000)
/usr/local/lib/python3.13/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)



--- Transcription brute Whisper ---
Efk-iyi-d afus-ik, ad k-qqneɣ taxa tmett-ik.

--- Phrase complète corrigée ---
Efk-iyi-d afus-ik, ad qqneɣ tax tmetti.


In [17]:
import shutil
import torch
from google.colab import drive, files
from transformers import WhisperProcessor, WhisperForConditionalGeneration
from peft import PeftModel

drive.mount("/content/drive")

lora_dir = "/content/drive/MyDrive/whisper-kabyle-full"
merged_dir = "/content/drive/MyDrive/whisper-kabyle-final-merged"

# 1. Fusionner Whisper-small + votre entraînement kabyle en un seul modèle indépendant
base_model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-small")
peft_model = PeftModel.from_pretrained(base_model, lora_dir)
merged_model = peft_model.merge_and_unload()

# 2. Sauvegarder le modèle complet dans Google Drive (~970 Mo)
merged_model.save_pretrained(merged_dir)
processor = WhisperProcessor.from_pretrained(lora_dir)
processor.save_pretrained(merged_dir)
print("Modèle complet fusionné sauvegardé dans Google Drive :", merged_dir)

# 3. Créer une archive .zip pour le télécharger aussi sur votre ordinateur (facultatif)
shutil.make_archive("/content/whisper-kabyle-final", "zip", merged_dir)
files.download("/content/whisper-kabyle-final.zip")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Modèle complet fusionné sauvegardé dans Google Drive : /content/drive/MyDrive/whisper-kabyle-final-merged


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [18]:
import shutil
import torch
from google.colab import drive, files
from transformers import WhisperProcessor, WhisperForConditionalGeneration
from peft import PeftModel

drive.mount("/content/drive")

lora_dir = "/content/drive/MyDrive/whisper-kabyle-full"
merged_dir = "/content/drive/MyDrive/whisper-kabyle-final-merged"

# 1. Fusionner Whisper-small + votre entraînement kabyle en un seul modèle indépendant
base_model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-small")
peft_model = PeftModel.from_pretrained(base_model, lora_dir)
merged_model = peft_model.merge_and_unload()

# 2. Sauvegarder le modèle complet dans Google Drive (~970 Mo)
merged_model.save_pretrained(merged_dir)
processor = WhisperProcessor.from_pretrained(lora_dir)
processor.save_pretrained(merged_dir)
print("Modèle complet fusionné sauvegardé dans Google Drive :", merged_dir)

# 3. Créer une archive .zip pour le télécharger aussi sur votre ordinateur (facultatif)
shutil.make_archive("/content/whisper-kabyle-final", "zip", merged_dir)
files.download("/content/whisper-kabyle-final.zip")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Modèle complet fusionné sauvegardé dans Google Drive : /content/drive/MyDrive/whisper-kabyle-final-merged


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>